In [1]:
import pandas as pd

stroke = pd.read_csv("healthcare-dataset-stroke-data.csv")
house = pd.read_csv("kc_house_data.csv")

print("Stroke dataset loaded:", stroke.shape)
print("House dataset loaded:", house.shape)

Stroke dataset loaded: (5110, 12)
House dataset loaded: (21613, 21)


In [2]:
display(stroke.head())
display(house.head())

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [3]:
print("STROKE DATASET")
print("Shape:", stroke.shape)
print("Columns:", stroke.columns.tolist())
print("Missing values:\n", stroke.isnull().sum())

print("\nHOUSE PRICE DATASET")
print("Shape:", house.shape)
print("Columns:", house.columns.tolist())
print("Missing values:\n", house.isnull().sum())

STROKE DATASET
Shape: (5110, 12)
Columns: ['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']
Missing values:
 id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

HOUSE PRICE DATASET
Shape: (21613, 21)
Columns: ['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']
Missing values:
 id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
wate

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix
from sklearn.base import clone

# 1. Separate input features and target
data = stroke.dropna(subset=["stroke"]).copy()

X = data.drop(columns=["stroke", "id"], errors="ignore")
y = data["stroke"]

# 2. Identify numerical and categorical columns
numeric_cols = X.select_dtypes(include="number").columns
categorical_cols = X.select_dtypes(exclude="number").columns

# 3. Prepare numerical and categorical data
numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# 4. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Create and train the classification model
stroke_model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

stroke_model.fit(X_train, y_train)

# 6. Make predictions
y_pred = stroke_model.predict(X_test)

# 7. Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7455968688845401

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022

Confusion Matrix:
 [[722 250]
 [ 10  40]]


In [5]:
import joblib

joblib.dump(stroke_model, "stroke_model.joblib")

print("Classification model saved successfully!")

Classification model saved successfully!


In [6]:
# Separate features and target
house_data = house.dropna(subset=["price"]).copy()

X_house = house_data.drop(columns=["price"], errors="ignore")
y_house = house_data["price"]

# Remove ID column if present
X_house = X_house.drop(columns=["id"], errors="ignore")

# Identify numerical and categorical columns
num_cols = X_house.select_dtypes(include="number").columns
cat_cols = X_house.select_dtypes(exclude="number").columns

# Preprocess the data
num_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

cat_pipe = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

house_preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# Split into training and testing sets
X_train_house, X_test_house, y_train_house, y_test_house = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

# Create and train regression model
from sklearn.linear_model import LinearRegression

house_model = make_pipeline(
    house_preprocessor,
    LinearRegression()
)

house_model.fit(X_train_house, y_train_house)

print("Regression model trained successfully!")

Regression model trained successfully!


In [7]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Predict house prices
y_pred_house = house_model.predict(X_test_house)

# Evaluate the model
print("R² Score:", r2_score(y_test_house, y_pred_house))
print("Mean Absolute Error:", mean_absolute_error(y_test_house, y_pred_house))
print("Mean Squared Error:", mean_squared_error(y_test_house, y_pred_house))

R² Score: 0.6964529730414386
Mean Absolute Error: 129202.25028672782
Mean Squared Error: 45889241550.93873


In [8]:
joblib.dump(house_model, "house_model.joblib")

print("Regression model saved successfully!")

Regression model saved successfully!


In [9]:

%%writefile app.py
import pandas as pd
import joblib
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles

app = FastAPI(title="Machine Learning Prediction App")

# Load the saved models
stroke_model = joblib.load("stroke_model.joblib")
house_model = joblib.load("house_model.joblib")


@app.get("/", response_class=HTMLResponse)
def home():
    with open("index.html", "r", encoding="utf-8") as file:
        return file.read()


@app.post("/predict/stroke")
def predict_stroke(data: dict):
    features = pd.DataFrame([data])
    prediction = int(stroke_model.predict(features)[0])

    return {
        "prediction": prediction,
        "result": "Stroke predicted" if prediction == 1
                  else "No stroke predicted"
    }


@app.post("/predict/house")
def predict_house(data: dict):
    features = pd.DataFrame([data])
    prediction = float(house_model.predict(features)[0])

    return {
        "predicted_price": round(prediction, 2)
    }


Writing app.py


In [10]:

%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Machine Learning Prediction App</title>

    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 850px;
            margin: 30px auto;
            padding: 20px;
            background: #f3f6fa;
        }

        h1, h2 {
            color: #23436b;
        }

        .card {
            background: white;
            padding: 20px;
            margin: 20px 0;
            border-radius: 10px;
        }

        textarea {
            width: 100%;
            height: 240px;
            box-sizing: border-box;
            padding: 10px;
            font-family: monospace;
        }

        button {
            background: #23436b;
            color: white;
            border: none;
            padding: 12px 20px;
            cursor: pointer;
            border-radius: 5px;
        }

        .result {
            margin-top: 15px;
            font-weight: bold;
            overflow-wrap: anywhere;
        }
    </style>
</head>

<body>
    <h1>Machine Learning Prediction App</h1>
    <p>Choose a model and enter the input data as JSON.</p>

    <div class="card">
        <h2>1. Stroke Classification</h2>

        <p>Predicts stroke status from patient information.
        This is an educational model, not a medical diagnosis.</p>

        <textarea id="strokeData">{
  "gender": "Male",
  "age": 67,
  "hypertension": 0,
  "heart_disease": 1,
  "ever_married": "Yes",
  "work_type": "Private",
  "Residence_type": "Urban",
  "avg_glucose_level": 228.69,
  "bmi": 36.6,
  "smoking_status": "formerly smoked"
}</textarea>

        <button onclick="predict('stroke')">
            Predict Stroke
        </button>

        <div id="strokeResult" class="result"></div>
    </div>

    <div class="card">
        <h2>2. House Price Regression</h2>

        <p>Enter house features to estimate its price.</p>

        <textarea id="houseData">{
  "date": "20141013T000000",
  "bedrooms": 3,
  "bathrooms": 2.0,
  "sqft_living": 1800,
  "sqft_lot": 5000,
  "floors": 1,
  "waterfront": 0,
  "view": 0,
  "condition": 3,
  "grade": 7,
  "sqft_above": 1500,
  "sqft_basement": 300,
  "yr_built": 1990,
  "yr_renovated": 0,
  "zipcode": 98103,
  "lat": 47.5,
  "long": -122.3,
  "sqft_living15": 1800,
  "sqft_lot15": 5000
}</textarea>

        <button onclick="predict('house')">
            Predict House Price
        </button>

        <div id="houseResult" class="result"></div>
    </div>

    <script>
        async function predict(type) {
            const result = document.getElementById(type + "Result");
            const inputId = type === "stroke"
                ? "strokeData" : "houseData";

            try {
                const data = JSON.parse(
                    document.getElementById(inputId).value
                );

                result.textContent = "Getting prediction...";

                const response = await fetch(
                    "/predict/" + type,
                    {
                        method: "POST",
                        headers: {
                            "Content-Type": "application/json"
                        },
                        body: JSON.stringify(data)
                    }
                );

                const output = await response.json();

                if (!response.ok) {
                    throw new Error(
                        output.detail || "Prediction failed"
                    );
                }

                result.textContent = type === "stroke"
                    ? output.result
                    : "Predicted price: " +
                      output.predicted_price.toLocaleString();

            } catch (error) {
                result.textContent =
                    "Error: " + error.message;
            }
        }
    </script>
</body>
</html>


Writing index.html


In [5]:
import os
from pathlib import Path

print("Notebook's current folder:")
print(os.getcwd())

print("\nDoes the Desktop ml folder exist?")
print(Path(r"C:\Users\Aatikah\Desktop\ml").exists())

print("\nSearching your Desktop for both datasets...")

desktop = Path(r"C:\Users\Aatikah\Desktop")

for file in desktop.rglob("*.csv"):
    if file.name.lower() in [
        "healthcare-dataset-stroke-data.csv",
        "kc_house_data.csv"
    ]:
        print(file)

Notebook's current folder:
c:\Users\Aatikah\Desktop\ml\__pycache__

Does the Desktop ml folder exist?
True

Searching your Desktop for both datasets...
C:\Users\Aatikah\Desktop\ml\kc_house_data.csv
C:\Users\Aatikah\Desktop\ml theory\healthcare-dataset-stroke-data.csv


In [6]:
import pandas as pd

stroke = pd.read_csv(
    r"C:\Users\Aatikah\Desktop\ml theory\healthcare-dataset-stroke-data.csv"
)

house = pd.read_csv(
    r"C:\Users\Aatikah\Desktop\ml\kc_house_data.csv"
)

print("Stroke dataset shape:", stroke.shape)
print("House-price dataset shape:", house.shape)

print("\nStroke dataset preview:")
display(stroke.head())

print("\nHouse-price dataset preview:")
display(house.head())

Stroke dataset shape: (5110, 12)
House-price dataset shape: (21613, 21)

Stroke dataset preview:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1



House-price dataset preview:


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [8]:
from sklearn.model_selection import train_test_split

# Prepare stroke data
data = stroke.dropna(subset=["stroke"]).copy()

X = data.drop(columns=["stroke", "id"], errors="ignore")
y = data["stroke"]

# Identify column types
numeric_cols = X.select_dtypes(include="number").columns
categorical_cols = X.select_dtypes(exclude="number").columns

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining class distribution:")
print(y_train.value_counts())

Training rows: 4088
Testing rows: 1022

Training class distribution:
stroke
0    3889
1     199
Name: count, dtype: int64


In [13]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Preprocessing for numerical and categorical columns
def make_preprocessor():
    numeric_pipeline = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler()
    )

    categorical_pipeline = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    )

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

# Create the five classification models
models = {
    "Decision Tree - Gini": make_pipeline(
        make_preprocessor(),
        DecisionTreeClassifier(
            criterion="gini",
            class_weight="balanced",
            random_state=42
        )
    ),

    "Decision Tree - Entropy": make_pipeline(
        make_preprocessor(),
        DecisionTreeClassifier(
            criterion="entropy",
            class_weight="balanced",
            random_state=42
        )
    ),

    "Random Forest": make_pipeline(
        make_preprocessor(),
        RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    ),

    "SVM": make_pipeline(
        make_preprocessor(),
        SVC(
            kernel="rbf",
            class_weight="balanced"
        )
    ),

    "Naive Bayes": make_pipeline(
        make_preprocessor(),
        GaussianNB()
    )
}

# Train all five models
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "- trained successfully")


Decision Tree - Gini - trained successfully
Decision Tree - Entropy - trained successfully
Random Forest - trained successfully
SVM - trained successfully
Naive Bayes - trained successfully


In [10]:
# Fix and train only the Naive Bayes model
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline

models["Naive Bayes"] = make_pipeline(
    make_preprocessor(),
    GaussianNB()
)

models["Naive Bayes"].fit(X_train, y_train)

print("Naive Bayes - trained successfully")

Naive Bayes - trained successfully


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Include your existing Logistic Regression model, if available
all_models = {
    **models
}

results = []
predictions = {}

for name, model in all_models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Stroke Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Stroke Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "Stroke F1-score": f1_score(
            y_test, y_pred, zero_division=0
        )
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Stroke F1-score",
    ascending=False
).reset_index(drop=True)

print("MODEL COMPARISON")
display(results_df.round(3))

# Display confusion matrices
for name, y_pred in predictions.items():
    print("\n", "=" * 40)
    print(name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

MODEL COMPARISON


,Model,Accuracy,Stroke Precision,Stroke Recall,Stroke F1-score
0,SVM,0.762,0.133,0.70,0.224
1,Decision Tree - Gini,0.929,0.189,0.14,0.161
2,Naive Bayes,0.298,0.064,0.98,0.120
3,Decision Tree - Entropy,0.919,0.077,0.06,0.067
4,Random Forest,0.950,0.000,0.00,0.000



Decision Tree - Gini
Confusion Matrix:
[[942  30]
 [ 43   7]]

Decision Tree - Entropy
Confusion Matrix:
[[936  36]
 [ 47   3]]

Random Forest
Confusion Matrix:
[[971   1]
 [ 50   0]]

SVM
Confusion Matrix:
[[744 228]
 [ 15  35]]

Naive Bayes
Confusion Matrix:
[[256 716]
 [  1  49]]


In [14]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd

# Evaluate the five trained models
results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Stroke Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Stroke Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "Stroke F1-score": f1_score(
            y_test, y_pred, zero_division=0
        )
    })

# Display the comparison table, highest F1-score first
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Stroke F1-score",
    ascending=False
).reset_index(drop=True)

print("CLASSIFICATION MODEL COMPARISON")
display(results_df.round(3))

# Display a confusion matrix for each model
for name, y_pred in predictions.items():
    print("\n" + "=" * 45)
    print(name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


CLASSIFICATION MODEL COMPARISON


,Model,Accuracy,Stroke Precision,Stroke Recall,Stroke F1-score
0,SVM,0.762,0.133,0.70,0.224
1,Decision Tree - Gini,0.929,0.189,0.14,0.161
2,Naive Bayes,0.298,0.064,0.98,0.120
3,Decision Tree - Entropy,0.919,0.077,0.06,0.067
4,Random Forest,0.950,0.000,0.00,0.000



Decision Tree - Gini
Confusion Matrix:
[[942  30]
 [ 43   7]]

Decision Tree - Entropy
Confusion Matrix:
[[936  36]
 [ 47   3]]

Random Forest
Confusion Matrix:
[[971   1]
 [ 50   0]]

SVM
Confusion Matrix:
[[744 228]
 [ 15  35]]

Naive Bayes
Confusion Matrix:
[[256 716]
 [  1  49]]


In [15]:
print("HOUSE-PRICE REGRESSION MODEL RESULTS")

if "house_model" in globals():
    print("R² Score:", house_model.score(X_house_test, y_house_test))
else:
    print("House model variable not found. We will check the existing regression cells.")

HOUSE-PRICE REGRESSION MODEL RESULTS
House model variable not found. We will check the existing regression cells.


In [16]:
print("Variables related to house-price regression:")

for name in globals():
    if any(word in name.lower() for word in ["house", "regression", "linear", "r2"]):
        print(name)

Variables related to house-price regression:
LogisticRegression
house


In [17]:
print("House dataset shape:", house.shape)
print("\nColumn names:")
print(house.columns.tolist())

print("\nFirst 5 rows:")
display(house.head())

House dataset shape: (21613, 21)

Column names:
['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']

First 5 rows:


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [18]:
# STEP 1: HOUSE-PRICE REGRESSION MODEL

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Copy the dataset
house_data = house.copy()

# Remove rows with missing values
house_data = house_data.dropna()

# Remove columns that should not be used as prediction features
X_house = house_data.drop(columns=["price", "id", "date"])
y_house = house_data["price"]

# Split data into training and testing sets
X_house_train, X_house_test, y_house_train, y_house_test = train_test_split(
    X_house,
    y_house,
    test_size=0.2,
    random_state=42
)

# Train the Linear Regression model
house_regression_model = LinearRegression()
house_regression_model.fit(X_house_train, y_house_train)

# Make predictions
y_house_pred = house_regression_model.predict(X_house_test)

# Evaluate the model
print("HOUSE-PRICE REGRESSION RESULTS")
print("Training rows:", len(X_house_train))
print("Testing rows:", len(X_house_test))
print("R² Score:", round(r2_score(y_house_test, y_house_pred), 4))
print("MAE:", round(mean_absolute_error(y_house_test, y_house_pred), 2))
print("MSE:", round(mean_squared_error(y_house_test, y_house_pred), 2))

HOUSE-PRICE REGRESSION RESULTS
Training rows: 17290
Testing rows: 4323
R² Score: 0.7012
MAE: 127493.34
MSE: 45173046132.79


In [19]:
import joblib

joblib.dump(
    house_regression_model,
    "house_regression_model.pkl"
)

print("House-price regression model saved successfully.")

House-price regression model saved successfully.


In [20]:
import os

print("MODEL FILE CHECK")

for filename in [
    "house_regression_model.pkl",
    "stroke_model.pkl",
    "stroke_model.joblib",
    "house_model.pkl",
    "house_model.joblib"
]:
    if os.path.exists(filename):
        print("FOUND:", filename)

MODEL FILE CHECK
FOUND: house_regression_model.pkl


In [21]:
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nFiles in the current folder:")
for filename in os.listdir("."):
    if filename.endswith((".pkl", ".joblib", ".py", ".html")):
        print(filename)

Current notebook folder:
c:\Users\Aatikah\Desktop\ml\__pycache__

Files in the current folder:
house_regression_model.pkl


In [22]:
import os

project_folder = r"C:\Users\Aatikah\Desktop\ml"

print("PROJECT FOLDER FILES")

for filename in os.listdir(project_folder):
    full_path = os.path.join(project_folder, filename)
    if os.path.isfile(full_path):
        print(filename)
    elif os.path.isdir(full_path):
        print("[Folder]", filename)
        

PROJECT FOLDER FILES
app.py
house_model.joblib
index.html
kc_house_data.csv
readme.md
stroke_model.joblib
[Folder] __pycache__


In [23]:

# Check the evaluation metrics available in this notebook

print("=== MODELS AND EVALUATION RESULTS ===")

# Check which model variables exist
model_names = [
    "stroke_model",
    "house_model",
    "house_regression_model",
    "y_test",
    "y_pred",
    "y_house_test",
    "y_house_pred"
]

for name in model_names:
    if name in globals():
        print(f"{name}: Available")
    else:
        print(f"{name}: Not found")

print("\n=== SAVED MODEL FILES ===")

import os

for filename in ["stroke_model.joblib", "house_model.joblib"]:
    print(f"{filename}: {'Found' if os.path.exists(filename) else 'Not found'}")


=== MODELS AND EVALUATION RESULTS ===
stroke_model: Not found
house_model: Not found
house_regression_model: Available
y_test: Available
y_pred: Available
y_house_test: Available
y_house_pred: Available

=== SAVED MODEL FILES ===
stroke_model.joblib: Not found
house_model.joblib: Not found


In [24]:

import os
import joblib

project_folder = r"C:\Users\Aatikah\Desktop\ml"

for filename in ["stroke_model.joblib", "house_model.joblib"]:
    path = os.path.join(project_folder, filename)
    print(f"\nChecking: {filename}")

    if os.path.exists(path):
        model = joblib.load(path)
        print("Status: Loaded successfully")
        print("Model type:", type(model).__name__)
    else:
        print("Status: File not found")



Checking: stroke_model.joblib
Status: Loaded successfully
Model type: Pipeline

Checking: house_model.joblib
Status: Loaded successfully
Model type: Pipeline


In [26]:

import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_absolute_error

project_folder = r"C:\Users\Aatikah\Desktop\ml"

# Load the exact saved models used by the website
stroke_model = joblib.load(os.path.join(project_folder, "stroke_model.joblib"))
house_model = joblib.load(os.path.join(project_folder, "house_model.joblib"))

# Load the datasets
stroke = pd.read_csv(
    r"C:\Users\Aatikah\Desktop\ml theory\healthcare-dataset-stroke-data.csv"
)
house = pd.read_csv(os.path.join(project_folder, "kc_house_data.csv"))

# Prepare stroke test data
stroke = stroke.drop(columns=["id"])
X_stroke = stroke.drop(columns=["stroke"])
y_stroke = stroke["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X_stroke, y_stroke, test_size=0.2, random_state=42, stratify=y_stroke
)

print("STROKE MODEL")
print("Accuracy:", accuracy_score(y_test, stroke_model.predict(X_test)))
print(classification_report(y_test, stroke_model.predict(X_test), zero_division=0))

# Prepare house test data

# Prepare house test data, keeping the date column
house = pd.read_csv(
    os.path.join(project_folder, "kc_house_data.csv")
).dropna()

X_house = house.drop(columns=["price", "id"])
y_house = house["price"]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

house_predictions = house_model.predict(X_test_h)

print("\nHOUSE PRICE MODEL")
print("R²:", r2_score(y_test_h, house_predictions))
print("MAE:", mean_absolute_error(y_test_h, house_predictions))


STROKE MODEL
Accuracy: 0.7455968688845401
              precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022


HOUSE PRICE MODEL
R²: 0.6964529730414386
MAE: 129202.25028672782
